# DriftSense: Data Generation Pipeline (Phase D1 - D4)

This pipeline synthesizes **DriftBench-v1**:
- **D1**: World demand generation (multi-seasonality, negative-binomial noise, Indian calendar, Poisson deployments) & Infrastructure capacity / autoscaling telemetry across 6 services.
- **D2**: FOCUS pricing model (on-demand, committed, credits/refunds, contracted costs, partitioned parquet).
- **D3**: Fault injection at root telemetry layer (6 fault types, decoys, overlapping, confounded scenarios) & `ground_truth` generation.
- **D4**: Realistic data dirt injection (missing hours, duplicates, late restatements, timezone shifts, unit mix) & `dirt_log`.

### 1. Setup & Environment

In [ ]:
# Mount Drive for Colab persistent artifact storage
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import sys
from pathlib import Path

current_dir = Path.cwd()
if (current_dir / 'src').exists():
    sys.path.insert(0, str(current_dir / 'src'))
elif (current_dir.parent / 'src').exists():
    sys.path.insert(0, str(current_dir.parent / 'src'))

import numpy as np
import polars as pl
import matplotlib.pyplot as plt

import driftsense
from driftsense.sim.world import generate_world_timeline
from driftsense.sim.infra import generate_infra_telemetry
from driftsense.sim.pricing import apply_pricing_model, save_partitioned_billed_parquet, FOCUS_COLUMNS
from driftsense.sim.faults import inject_faults_and_decoys
from driftsense.utils.seed import set_seed
from driftsense.utils.io import load_config
from driftsense.utils.summary import print_result_summary

print(f"DriftSense Version: {driftsense.__version__}")

### 2. Configuration & Scale Selection

In [ ]:
import time
start_time = time.time()

base_cfg = load_config("base.yaml")
scale_cfg = load_config("scale.yaml")

# Select scale: 'tiny' for smoke/tests, 'small' for quick Colab verification, 'full' for 365-day benchmark
SCALE = "tiny"  # Switch to 'full' for production run in Colab
active_params = scale_cfg[SCALE]

SEED = base_cfg.get("seed", 42)
set_seed(SEED)

DAYS = active_params["days"]
PROVIDERS = base_cfg.get("providers", ["aws_like", "azure_like"])
RES_PER_PROVIDER = active_params["resources_per_provider"]

print(f"Active Scale: {SCALE} | Days: {DAYS} | Providers: {PROVIDERS} | Resources/Provider: {RES_PER_PROVIDER}")

### 3. Run: World, Infra, Fault Injection (D3) & FOCUS Pricing (D2)

In [ ]:
print("Generating world timeline (D1)...")
world_df, events_df = generate_world_timeline(
    days=DAYS,
    start_date_str=base_cfg["date_range"]["start"],
    annual_growth_rate=0.35,
    seed=SEED,
)

print("Generating infrastructure telemetry (D1)...")
clean_infra_df = generate_infra_telemetry(
    world_df=world_df,
    providers=PROVIDERS,
    resources_per_provider=RES_PER_PROVIDER,
    seed=SEED,
)

print("Injecting root-cause faults, decoys, and generating ground truth (D3)...")
faulted_infra_df, ground_truth_df, updated_events_df = inject_faults_and_decoys(
    infra_df=clean_infra_df,
    world_df=world_df,
    events_df=events_df,
    seed=SEED,
)

print("Applying FOCUS pricing and credits model (D2)...")
clean_truth_df, _ = apply_pricing_model(clean_infra_df, events_df=None, seed=SEED)
billed_df, final_events_df = apply_pricing_model(
    infra_df=faulted_infra_df,
    events_df=updated_events_df,
    credit_rate=0.005,
    seed=SEED,
)

print(f"World timeline shape: {world_df.shape}")
print(f"Clean truth shape: {clean_truth_df.shape}")
print(f"Faulted billed shape: {billed_df.shape}")
print(f"Ground truth incidents & decoys: {len(ground_truth_df)}")
print(f"Events logged: {len(final_events_df)}")

### 4. Inspect: Telemetry Visualizations & Incident Overlays

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 13))

# 1. Web Traffic (Hourly Requests)
ts_world = world_df["timestamp"].to_numpy()
req_world = world_df["traffic_requests"].to_numpy()
limit_idx = min(len(ts_world), 24 * 14)
axes[0].plot(ts_world[:limit_idx], req_world[:limit_idx], color="royalblue", lw=1.2)
axes[0].set_title("World Web Traffic (Hourly Requests)", fontsize=11, fontweight="bold")
axes[0].set_ylabel("Requests / hr")
axes[0].grid(True, alpha=0.3)

# 2. Hourly Billed Cost with Ground Truth Incident Windows Overlay
service_cost_ts = billed_df.group_by(["ChargePeriodStart", "ServiceName"]).agg(pl.col("BilledCost").sum()).sort("ChargePeriodStart")
for s in service_cost_ts["ServiceName"].unique():
    sub_s = service_cost_ts.filter(pl.col("ServiceName") == s).head(24 * 14)
    axes[1].plot(sub_s["ChargePeriodStart"].to_numpy(), sub_s["BilledCost"].to_numpy(), label=s, lw=1.2)

# Highlight non-decoy incident windows
non_decoys = ground_truth_df.filter(~pl.col("is_decoy"))
for inc in non_decoys.iter_rows(named=True):
    axes[1].axvspan(inc["start"], inc["end"], color="red", alpha=0.15, label="Fault Window" if inc["incident_id"] == non_decoys["incident_id"][0] else "")

axes[1].set_title("Hourly Cost with Ground Truth Incident Windows (Red Shading)", fontsize=11, fontweight="bold")
axes[1].set_ylabel("Cost (INR)")
axes[1].legend(loc="upper right", ncol=4)
axes[1].grid(True, alpha=0.3)

# 3. Compute Service CPU with Autoscaling Dynamics
comp_sub = billed_df.filter(pl.col("ServiceName") == "compute").head(24 * 14)
axes[2].plot(comp_sub["ChargePeriodStart"].to_numpy(), comp_sub["x_cpu_pct"].to_numpy(), color="crimson", label="CPU %")
axes[2].set_title("Compute Service CPU Utilization with Injected Anomalies", fontsize=11, fontweight="bold")
axes[2].set_ylabel("CPU %")
axes[2].grid(True, alpha=0.3)

# 4. Ground Truth Incidents Summary Table
axes[3].axis('off')
table_data = [[r["incident_id"], str(r["start"])[:16], str(r["end"])[:16], r["service"], r["root_cause"], "Decoy" if r["is_decoy"] else "Fault"] for r in ground_truth_df.head(6).iter_rows(named=True)]
col_labels = ["ID", "Start", "End", "Service", "Root Cause", "Category"]
table = axes[3].table(cellText=table_data, colLabels=col_labels, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.4)
axes[3].set_title("Ground Truth Scenarios Sample", fontsize=11, fontweight="bold", pad=20)

plt.tight_layout()
plt.show()

### 5. Checks: Acceptance Criteria Verification
- Cost deviation >= 2.5 robust sigma on all non-decoy incidents
- Decoys flagged with `is_decoy=True`
- Held-out fault (`egress_spike`) absent from train and calibration months
- FOCUS columns and zero NaNs

In [ ]:
checks_status = {}

# 1. Held-out fault invariant
total_hours = len(world_df)
cal_cutoff_idx = int(total_hours * 0.80)
all_timestamps = world_df["timestamp"].to_list()
cal_cutoff_dt = all_timestamps[min(cal_cutoff_idx, total_hours - 1)]
egress_incidents = ground_truth_df.filter(pl.col("root_cause") == "egress_spike")
assert len(egress_incidents) > 0
assert all(s >= cal_cutoff_dt for s in egress_incidents["start"])
checks_status["held_out_fault_restricted"] = "PASS"

# 2. Decoys present
decoys = ground_truth_df.filter(pl.col("is_decoy"))
assert len(decoys) > 0
checks_status["decoys_present"] = "PASS"

# 3. Cost deviation on non-decoy incidents
clean_srv_grp = clean_truth_df.group_by(["ChargePeriodStart", "ServiceName"]).agg(pl.col("BilledCost").sum())
fault_srv_grp = billed_df.group_by(["ChargePeriodStart", "ServiceName"]).agg(pl.col("BilledCost").sum())

for inc in ground_truth_df.filter(~pl.col("is_decoy")).iter_rows(named=True):
    srv = inc["service"]
    clean_costs = clean_srv_grp.filter(pl.col("ServiceName") == srv)["BilledCost"].to_numpy()
    med = np.median(clean_costs)
    mad = max(1.0, 1.4826 * np.median(np.abs(clean_costs - med)))
    window_max = fault_srv_grp.filter(
        (pl.col("ServiceName") == srv) & (pl.col("ChargePeriodStart") >= inc["start"]) & (pl.col("ChargePeriodStart") <= inc["end"])
    )["BilledCost"].max()
    z = (window_max - med) / mad
    assert z >= 2.5, f"Weak z-score {z} for {inc['incident_id']}"
checks_status["fault_cost_deviations_significant"] = "PASS"

# 4. Zero NaNs
assert all(billed_df[col].null_count() == 0 for col in billed_df.columns)
checks_status["zero_nans"] = "PASS"

print("Acceptance Checks Status:", checks_status)

### 6. Save Artifacts

In [ ]:
out_dir = Path("artifacts/d3")
out_dir.mkdir(parents=True, exist_ok=True)

gt_path = out_dir / "ground_truth.parquet"
clean_truth_path = out_dir / "clean_truth.parquet"
events_path = out_dir / "events.parquet"

ground_truth_df.write_parquet(gt_path)
clean_truth_df.write_parquet(clean_truth_path)
final_events_df.write_parquet(events_path)

saved_partitions = save_partitioned_billed_parquet(billed_df, output_dir="data/raw")
artifacts_saved = [str(gt_path), str(clean_truth_path), str(events_path)] + saved_partitions[:3]
print("Saved artifacts successfully:", artifacts_saved)

### 7. RESULT SUMMARY

In [ ]:
runtime = time.time() - start_time

summary = {
    "phase": "D3",
    "scale": SCALE,
    "seed": SEED,
    "runtime_sec": runtime,
    "shapes": {
        "billed_df": f"{billed_df.height} x {billed_df.width}",
        "clean_truth_df": f"{clean_truth_df.height} x {clean_truth_df.width}",
        "ground_truth_df": f"{ground_truth_df.height} x {ground_truth_df.width}",
        "events_df": f"{final_events_df.height} x {final_events_df.width}",
    },
    "key_metrics": {
        "total_incidents": len(ground_truth_df),
        "num_real_faults": len(ground_truth_df.filter(~pl.col("is_decoy"))),
        "num_decoys": len(ground_truth_df.filter(pl.col("is_decoy"))),
        "held_out_faults": len(egress_incidents),
    },
    "checks": checks_status,
    "warnings": [],
    "artifacts": artifacts_saved,
}

print_result_summary(summary)